In [1]:
import glob
import json
import re
from xml.etree import ElementTree as ET
# from src.data.xml_extraction import gen_xml_paths

In [2]:
field_keywords = json.load(open("../data/external/field_keywords.json", encoding="utf8"))

In [5]:
def gen_xml_paths(path: str | os.PathLike) -> list[str]:
    """
    Collect xmls from a path
    Retries needed as this was originally on a network path that failed occasionally
    :param path: str : A location of transkribus model output xmls
    :return: list[str], list[str]
    """

    attempts = 0
    while attempts < 3:
        xmls = glob.glob(path)
        if xmls:
            break
        else:
            attempts += 1
            continue
    else:
        raise IOError(f"Failed to connect to {path}")

    return xmls

In [6]:
def parse_custom_attribute_string(element: Element) -> list[tuple[str, tuple[str, str]]]:
    """
    Parse the custom attributes of an XML element
    Convert the custom string into a list of (Transkribus) tags and tag values

    Args:
        element (Element): _description_

    Returns:
        list[tuple[str, tuple[str, str]]]: _description_
    """
    attributes_raw = element.attrib.get("custom")
    # Handle misformatted Unicode, U+0020 (space), U+0027 (apostrophe)
    attributes = attributes_raw.replace(r"\u0020", " ").replace(r"\u0027", "'")
    attrib_pair_re = re.compile(r"(?P<tag>\w+) (?P<text>\{[\.\w\s:;\d\\'’-]+\})")
    attrib_inner_re = re.compile(r"(?P<tag>\w+):(?P<text>[\.\w\s\d\\'’-]+)")
    all_attribs = attrib_pair_re.findall(attributes)

    inner_found = [(k, attrib_inner_re.findall(v[1:-1])) for k,v in all_attribs]
    # breakpoint()
    return inner_found

In [7]:
xmls = gen_xml_paths("../data/raw/BMC_11_2/13470627/*.xml")

In [28]:
tree = ET.parse(xmls[1])

In [29]:
root = tree.getroot()

In [30]:
[print(parse_custom_attribute_string(e)) for e in root[1][2:]]

[('readingOrder', [('index', '0')]), ('structure', [('type', 'Binding')])]
[('readingOrder', [('index', '1')]), ('structure', [('type', 'Provenance')])]
[('readingOrder', [('index', '2')]), ('structure', [('type', 'Dating')])]
[('readingOrder', [('index', '3')]), ('structure', [('type', 'Dating')])]
[('readingOrder', [('index', '4')]), ('structure', [('type', 'table')])]
[('readingOrder', [('index', '5')]), ('structure', [('type', 'Binding')])]
[('readingOrder', [('index', '6')]), ('structure', [('type', 'Provenance')])]


[None, None, None, None, None, None, None]

In [31]:
field_keywords

{'Binding': ['Binding', 'Bound', 'Rebound', 'Inlaid'],
 'Dating': ['Dating'],
 'Provenance': ['Provenance',
  'Presented',
  'From',
  'Grenville Copy',
  'King George III’s Copy',
  'Bought']}

In [32]:
field_keywords.get('Dating', None)

['Dating']

- iterate over all xmls
- iterate over all regions in each xml
- get the structure type for that region
- check if region is one of binding, dating, provenance
- if yes check if region starts with one of the field keywords
- if yes assign to good list
- if no assign to bad list

- iterate over regions in one xml
- get the structure type for that region
- check if region is one of binding, dating, provenance
- if yes check get keywords for field
- get all text for region
- process text to remove blank lines
- check region starts with one of the field keywords
- create good list/bad list
- assign to good/list bad list
- generalise to iterate over all xmls

In [33]:
[print(parse_custom_attribute_string(e)) for e in root[1][2:]]

[('readingOrder', [('index', '0')]), ('structure', [('type', 'Binding')])]
[('readingOrder', [('index', '1')]), ('structure', [('type', 'Provenance')])]
[('readingOrder', [('index', '2')]), ('structure', [('type', 'Dating')])]
[('readingOrder', [('index', '3')]), ('structure', [('type', 'Dating')])]
[('readingOrder', [('index', '4')]), ('structure', [('type', 'table')])]
[('readingOrder', [('index', '5')]), ('structure', [('type', 'Binding')])]
[('readingOrder', [('index', '6')]), ('structure', [('type', 'Provenance')])]


[None, None, None, None, None, None, None]

In [34]:
good_regions = []
bad_regions = []
# line below is iterating over the xml 
# for each iteration lets us access each text region 
for region in root[1][2:]:
    # line below getting the custom attribute string and structuring it and assigning it to attributes
    attributes = parse_custom_attribute_string(region)
    # line below is printing field type for each text region
    # print(attributes[1][1][0][1])
    # line below assigns field type to region_field_type  
    region_field_type = attributes[1][1][0][1]
    # line below assigns field names for OCR that we want to add to MARC records to target_field_types
    target_field_types = ['Binding', 'Dating', 'Provenance']
    # looks for target_field_types in the region_field_type
    if region_field_type in target_field_types:
        # assigns field key words to target_field_keywords
        target_field_keywords = field_keywords.get(region_field_type)
        # creates list to gather text lines for region
        region_lines = []
        # iterating over the text lines in each text region
        for line in region[1:-1]:
            # getting the text
            line_text = line[2][0].text
            # adding the text for the line to the list of region lines
            region_lines.append(line_text)
        
        non_blank_lines = []
        for line in region_lines:
            if line:
                non_blank_lines.append(line)
        for line in non_blank_lines[:3]:
            for kw in target_field_keywords:
                if kw in line:
                    good_regions.append(region)
                    break
        else: 
            bad_regions.append(region)
        
print(f"Good regions: {good_regions}")
print(f"Bad regions: {bad_regions}")

Good regions: [<Element '{http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15}TextRegion' at 0x000001FD81C531F0>, <Element '{http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15}TextRegion' at 0x000001FD81C49E90>, <Element '{http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15}TextRegion' at 0x000001FD81C49030>, <Element '{http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15}TextRegion' at 0x000001FD81C49030>]
Bad regions: [<Element '{http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15}TextRegion' at 0x000001FD81C531F0>, <Element '{http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15}TextRegion' at 0x000001FD81C526B0>, <Element '{http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15}TextRegion' at 0x000001FD81C51670>, <Element '{http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15}TextRegion' at 0x000001FD81C50C70>, <Element '{http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15}TextRegi